# Home Credit — EDA Part 2 (Rewritten, Consistent Version)

## Stage definition, internal-history anchor validation, collection signal extraction, and practical feature shortlist

Mục tiêu của notebook này:

- đọc output từ **EDA Part 1**
- build lại **stage assignment A / B / C** theo đúng business logic
- tránh dùng anchor sai nghĩa như `birth`, `employedfrom`
- profile nhanh các raw variables quan trọng để chuẩn bị cho notebook FE / modeling
- export artifact ổn định cho notebook tiếp theo

## 1. Config & load outputs from EDA Part 1

In [1]:

from __future__ import annotations

import csv
import re
from pathlib import Path
from typing import Iterable
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(140)
pl.Config.set_tbl_width_chars(260)

def first_existing(paths: list[str]) -> Path:
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    raise FileNotFoundError("Cannot detect path from candidates:\n" + "\n".join(paths))

TRAIN_DIR = first_existing([
    "/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train",
    "/kaggle/input/home-credit-credit-risk-model-stability/csv_files/train",
    "/kaggle/input/csv_files/train",
])

NB1_WORK_DIR = first_existing([
    "/kaggle/working/homecredit_proxy_notebook_01",
    "/kaggle/input/notebooks/hongvittrnh/homecredit-eda-part1/homecredit_proxy_notebook_01",
    "/mnt/data",
])

WORK_DIR = Path("/kaggle/working/homecredit_proxy_notebook_02")
WORK_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_CATALOG_PATH = NB1_WORK_DIR / "feature_catalog.parquet"
PROXY_CATALOG_PATH   = NB1_WORK_DIR / "proxy_candidate_catalog.parquet"
HEADER_INV_PATH      = NB1_WORK_DIR / "header_inventory.parquet"
FILE_CATALOG_PATH    = NB1_WORK_DIR / "file_catalog.parquet"

required_paths = [FEATURE_CATALOG_PATH, PROXY_CATALOG_PATH, HEADER_INV_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Run EDA Part 1 first. Missing files:\n" + "\n".join(missing))

feature_catalog = pl.read_parquet(FEATURE_CATALOG_PATH)
proxy_candidate_catalog = pl.read_parquet(PROXY_CATALOG_PATH)
header_inventory = pl.read_parquet(HEADER_INV_PATH)
file_catalog = pl.read_parquet(FILE_CATALOG_PATH) if FILE_CATALOG_PATH.exists() else None

print("TRAIN_DIR              :", TRAIN_DIR)
print("NB1_WORK_DIR           :", NB1_WORK_DIR)
print("WORK_DIR               :", WORK_DIR)
print("feature_catalog shape  :", feature_catalog.shape)
print("proxy_catalog shape    :", proxy_candidate_catalog.shape)
print("header_inventory shape :", header_inventory.shape)
display(feature_catalog.head(10))

TRAIN_DIR              : /kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train
NB1_WORK_DIR           : /kaggle/input/notebooks/hongvittrnh/homecredit-eda-part1/homecredit_proxy_notebook_01
WORK_DIR               : /kaggle/working/homecredit_proxy_notebook_02
feature_catalog shape  : (472, 25)
proxy_catalog shape    : (276, 14)
header_inventory shape : (1139, 4)


variable,description,n_tables,source_tables,source_files,exists_in_raw,suffix_type,suffix_id,source_group,feature_family,…,seed_stage_candidate,seed_priority,seed_note,proxy_concept_final,stage_candidate_final,priority_final,review_note,business_validated_flag,inference_availability,recommended_for_modeling
str,str,u32,list[str],list[str],bool,str,i64,str,str,…,str,str,str,str,str,str,str,bool,str,bool
"""MONTH""","""""",1,"[""train_base""]","[""train_base.csv""]",true,null,null,"""other""","""date_or_recency""",…,null,null,null,"""recency_or_tenure""","""review_first""","""low""","""""",false,"""review""",false
"""WEEK_NUM""","""""",1,"[""train_base""]","[""train_base.csv""]",true,null,null,"""other""","""date_or_recency""",…,null,null,null,"""recency_or_tenure""","""review_first""","""low""","""""",false,"""review""",false
"""actualdpd_943P""","""Days Past Due (DPD) of previous contract (actual).""",2,"[""train_applprev_1_0"", ""train_applprev_1_1""]","[""train_applprev_1_0.csv"", ""train_applprev_1_1.csv""]",true,"""P""",943,"""previous_application""","""delinquency""",…,"""B_or_C""","""high""","""current DPD""","""delinquency_risk""","""B_or_C""","""high""","""current DPD""",true,"""batch_scoring_ok""",true
"""actualdpdtolerance_344P""","""DPD of client with tolerance.""",2,"[""train_static_0_0"", ""train_static_0_1""]","[""train_static_0_0.csv"", ""train_static_0_1.csv""]",true,"""P""",344,"""application_static""","""delinquency""",…,null,null,null,"""delinquency_risk""","""B_or_C""","""high""","""""",true,"""batch_scoring_ok""",true
"""addres_district_368M""","""District of the person's address.""",1,"[""train_person_2""]","[""train_person_2.csv""]",true,"""M""",368,"""person""","""identity_stability""",…,null,null,null,"""identity_instability""","""A_or_B""","""medium""","""""",true,"""batch_scoring_ok""",true
"""addres_role_871L""","""Role of person's address.""",1,"[""train_person_2""]","[""train_person_2.csv""]",true,"""L""",871,"""person""","""identity_stability""",…,null,null,null,"""identity_instability""","""A_or_B""","""medium""","""""",true,"""batch_scoring_ok""",true
"""addres_zip_823M""","""Zip code of the address.""",1,"[""train_person_2""]","[""train_person_2.csv""]",true,"""M""",823,"""person""","""identity_stability""",…,null,null,null,"""identity_instability""","""A_or_B""","""medium""","""""",true,"""batch_scoring_ok""",true
"""amount_1115A""","""Credit amount of the active contract provided by the credit bureau.""",1,"[""train_credit_bureau_b_1""]","[""train_credit_bureau_b_1.csv""]",true,"""A""",1115,"""bureau""","""capacity""",…,"""A_or_B""","""high""","""active debt amount""","""credit_burden""","""A_or_B""","""high""","""active debt amount""",true,"""batch_scoring_ok""",true
"""amount_416A""","""Deposit amount.""",1,"[""train_deposit_1""]","[""train_deposit_1.csv""]",true,"""A""",416,"""deposit""","""capacity""",…,null,null,null,"""capacity_to_repay""","""A_or_B""","""high""","""""",true,"""batch_scoring_ok""",true


## 2. Helper functions

In [2]:

def read_csv_header(path: str | Path) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    return next(csv.reader([first_line]))

def list_train_files(base_dir: Path) -> pl.DataFrame:
    rows = []
    for p in sorted(base_dir.glob("*.csv")):
        rows.append({
            "file_name": p.name,
            "file_path": str(p),
            "table_tag": p.stem,
        })
    if not rows:
        raise FileNotFoundError(f"No csv files under {base_dir}")
    return pl.DataFrame(rows)

def first_existing_table_by_patterns(patterns: list[str]) -> Path | None:
    train_files_local = list_train_files(TRAIN_DIR)
    for pat in patterns:
        hit = train_files_local.filter(pl.col("table_tag").str.contains(pat, literal=False))
        if hit.height > 0:
            return Path(hit[0, "file_path"])
    return None

def detect_case_id_column(columns: list[str]) -> str | None:
    for c in ["case_id", "CASE_ID", "sk_id_curr", "SK_ID_CURR"]:
        if c in columns:
            return c
    return None

def detect_decision_date_column(columns: list[str]) -> str | None:
    for c in ["date_decision", "DATE_DECISION", "decision_date", "DecisionDate"]:
        if c in columns:
            return c
    return None

def detect_target_column(columns: list[str]) -> str | None:
    for c in ["target", "TARGET"]:
        if c in columns:
            return c
    return None

def parse_to_date_expr(col: str) -> pl.Expr:
    return pl.coalesce([
        pl.col(col).cast(pl.String, strict=False).str.strptime(pl.Date, "%Y-%m-%d", strict=False),
        pl.col(col).cast(pl.String, strict=False).str.strptime(pl.Date, "%Y-%m-%d %H:%M:%S", strict=False),
        pl.col(col).cast(pl.String, strict=False).str.strptime(pl.Date, "%d-%m-%Y", strict=False),
        pl.col(col).cast(pl.Date, strict=False),
    ])

def approx_months_from_days_expr(col: str) -> pl.Expr:
    return (pl.col(col) / 30.4375).round(2)

def safe_numeric_cast_expr(col: str) -> pl.Expr:
    return pl.col(col).cast(pl.Float64, strict=False)

def choose_first_existing(vars_: list[str], available_cols: list[str]) -> str | None:
    for v in vars_:
        if v in available_cols:
            return v
    return None

def collapse_duplicate_suffix_columns(df: pl.DataFrame, suffix: str = "__dup") -> pl.DataFrame:
    dup_cols = [c for c in df.columns if c.endswith(suffix)]
    if not dup_cols:
        return df

    base_names = sorted({c.replace(suffix, "") for c in dup_cols})
    exprs = []
    for base in base_names:
        candidates = []
        if base in df.columns:
            candidates.append(pl.col(base))
        dup_name = f"{base}{suffix}"
        if dup_name in df.columns:
            candidates.append(pl.col(dup_name))
        if candidates:
            exprs.append(pl.coalesce(candidates).alias(base))

    return df.with_columns(exprs).drop(dup_cols)

train_files = list_train_files(TRAIN_DIR)
display(train_files.head(10))

file_name,file_path,table_tag
str,str,str
"""train_applprev_1_0.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv""","""train_applprev_1_0"""
"""train_applprev_1_1.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_1.csv""","""train_applprev_1_1"""
"""train_applprev_2.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_2.csv""","""train_applprev_2"""
"""train_base.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_base.csv""","""train_base"""
"""train_credit_bureau_a_1_0.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_1_0.csv""","""train_credit_bureau_a_1_0"""
"""train_credit_bureau_a_1_1.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_1_1.csv""","""train_credit_bureau_a_1_1"""
"""train_credit_bureau_a_1_2.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_1_2.csv""","""train_credit_bureau_a_1_2"""
"""train_credit_bureau_a_1_3.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_1_3.csv""","""train_credit_bureau_a_1_3"""
"""train_credit_bureau_a_2_0.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_credit_bureau_a_2_0.csv""","""train_credit_bureau_a_2_0"""


## 3. Load base application table

Base table dùng để lấy:
- `case_id`
- `decision_date`
- `target` nếu có

In [3]:

base_path = first_existing_table_by_patterns([
    r"train_base",
    r"base",
    r"train_static_0",
    r"static_0",
])

if base_path is None:
    raise FileNotFoundError("Could not detect base/static file containing case_id and decision date.")

base_header = read_csv_header(base_path)
CASE_ID_COL = detect_case_id_column(base_header)
DECISION_DATE_COL = detect_decision_date_column(base_header)
TARGET_COL = detect_target_column(base_header)

if CASE_ID_COL is None:
    raise ValueError(f"Could not detect case_id column in {base_path.name}")
if DECISION_DATE_COL is None:
    raise ValueError(f"Could not detect decision date column in {base_path.name}")

read_cols = [CASE_ID_COL, DECISION_DATE_COL]
if TARGET_COL is not None:
    read_cols.append(TARGET_COL)

base_df = (
    pl.read_csv(base_path, columns=read_cols)
    .rename({
        CASE_ID_COL: "case_id",
        DECISION_DATE_COL: "decision_date_raw",
        **({TARGET_COL: "target"} if TARGET_COL is not None else {}),
    })
    .with_columns(parse_to_date_expr("decision_date_raw").alias("decision_date"))
    .drop("decision_date_raw")
)

print("base_path:", base_path)
print("base_df shape:", base_df.shape)
display(base_df.head(10))

base_path: /kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_base.csv
base_df shape: (1526659, 3)


case_id,target,decision_date
i64,i64,date
0,0,2019-01-03
1,0,2019-01-03
2,0,2019-01-04
3,0,2019-01-03
4,1,2019-01-04
5,0,2019-01-02
6,0,2019-01-03
7,0,2019-01-03
8,0,2019-01-03


## 4. Build a reusable header-path map

Map này giúp:
- biết variable nằm ở file nào
- join giữa `header_inventory` và `train_files`
- tái sử dụng cho anchor scan, collection scan, profiler

In [4]:

header_with_paths = (
    header_inventory
    .rename({"column_name": "variable"})
    .join(
        train_files.select(["file_name", "file_path", "table_tag"]),
        on=["file_name", "table_tag"],
        how="left",
    )
)

print("header_with_paths shape:", header_with_paths.shape)
display(header_with_paths.head(20))

header_with_paths shape: (1139, 5)


file_name,table_tag,variable,col_order,file_path
str,str,str,i64,str
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""case_id""",0,"""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv"""
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""actualdpd_943P""",1,"""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv"""
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""annuity_853A""",2,"""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv"""
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""approvaldate_319D""",3,"""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv"""
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""byoccupationinc_3656910L""",4,"""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv"""
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""cancelreason_3545846M""",5,"""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv"""
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""childnum_21L""",6,"""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv"""
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""creationdate_885D""",7,"""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv"""
"""train_applprev_1_0.csv""","""train_applprev_1_0""","""credacc_actualbalance_314A""",8,"""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv"""


## 5. Internal-history anchor screening

Ở bản rewrite này, anchor chỉ được chọn nếu nó phản ánh **internal Home Credit history**.

Logic chính:
- ưu tiên source kiểu `previous_application`, `applprev`, `prev`
- date phải mang nghĩa như `approval`, `activated`, `disbursement`, `application`, `origination`
- loại bỏ các date sai nghĩa business:
  - `birth`
  - `employ`
  - tax/person registry dates
  - DPD / closure / recovery dates

In [5]:

internal_anchor_catalog = (
    feature_catalog
    .filter(pl.col("exists_in_raw") == True)
    .with_columns([
        pl.col("variable").fill_null("").str.to_lowercase().alias("var_l"),
        pl.col("description").fill_null("").str.to_lowercase().alias("desc_l"),
        pl.col("source_group").fill_null("").str.to_lowercase().alias("src_group_l"),
        pl.when(pl.col("source_tables").is_null())
          .then(pl.lit(""))
          .otherwise(pl.col("source_tables").list.join(" | "))
          .alias("source_tables_str"),
    ])
    .with_columns([
        pl.col("source_tables_str").str.to_lowercase().alias("src_tables_l"),
    ])
    .with_columns([
        pl.concat_str(
            [
                pl.col("var_l"),
                pl.lit(" "),
                pl.col("desc_l"),
                pl.lit(" "),
                pl.col("src_group_l"),
                pl.lit(" "),
                pl.col("src_tables_l"),
            ],
            separator=""
        ).alias("text_l")
    ])
    .with_columns([
        (
            pl.col("src_group_l").str.contains(r"previous_application", literal=False)
            | pl.col("src_tables_l").str.contains(r"applprev|prev", literal=False)
        ).alias("is_internal_prevapp_source"),

        (
            pl.col("text_l").str.contains(
                r"first.*date|approval.*date|activated.*date|creation.*date|disbursement.*date|opening.*date|origination.*date|offer.*date|application.*date",
                literal=False
            )
        ).alias("good_internal_date_signal"),

        (
            pl.col("text_l").str.contains(
                r"birth|birthday|employ|employedfrom|tax|person|dpd|delinq|overdue|late|closure|closed|recovery|write.?off|target|week_num",
                literal=False
            )
        ).alias("bad_internal_anchor_signal"),
    ])
    .with_columns([
        (
            pl.col("is_internal_prevapp_source")
            & pl.col("good_internal_date_signal")
            & (~pl.col("bad_internal_anchor_signal"))
        ).alias("is_valid_internal_anchor")
    ])
    .sort(["is_valid_internal_anchor", "variable"], descending=[True, False])
)

display(
    internal_anchor_catalog
    .filter(pl.col("is_valid_internal_anchor") == True)
    .select([
        "variable", "description", "source_group", "source_tables",
        "priority", "leakage_risk"
    ])
    .head(40)
)

variable,description,source_group,source_tables,priority,leakage_risk
str,str,str,list[str],str,str
"""approvaldate_319D""","""Approval Date of Previous Application""","""previous_application""","[""train_applprev_1_0"", ""train_applprev_1_1""]","""medium""","""medium_review"""
"""creationdate_885D""","""Date when previous application was created.""","""previous_application""","[""train_applprev_1_0"", ""train_applprev_1_1""]","""medium""","""medium_review"""
"""dateactivated_425D""","""Contract activation date of the applicant's previous application.""","""previous_application""","[""train_applprev_1_0"", ""train_applprev_1_1""]","""high""","""medium_review"""
"""firstnonzeroinstldate_307D""","""Date of first instalment in the previous application.""","""previous_application""","[""train_applprev_1_0"", ""train_applprev_1_1""]","""medium""","""medium_review"""


## 6. Choose internal anchor columns for stage assignment

Có 2 mode:
- auto detect từ metadata
- manual override nếu bạn muốn khóa cứng danh sách

In [6]:

EXCLUDE_INTERNAL_ANCHOR_COLUMNS = {
    "WEEK_NUM",
    "week_num",
    "case_id",
    "CASE_ID",
    "target",
    "TARGET",
    "MONTH",
    "month",
    "num_group1",
    "num_group2",
}

AUTO_INTERNAL_ANCHOR_COLUMNS = (
    internal_anchor_catalog
    .filter(
        (pl.col("is_valid_internal_anchor") == True) &
        (pl.col("leakage_risk") != "high_review") &
        (~pl.col("variable").is_in(list(EXCLUDE_INTERNAL_ANCHOR_COLUMNS)))
    )
    .select("variable")
    .unique()
    .head(12)
    .to_series()
    .to_list()
)

# Có thể override thủ công nếu cần ổn định tuyệt đối giữa các lần chạy
MANUAL_INTERNAL_ANCHOR_COLUMNS: list[str] = []

INTERNAL_ANCHOR_COLUMNS = (
    MANUAL_INTERNAL_ANCHOR_COLUMNS
    if len(MANUAL_INTERNAL_ANCHOR_COLUMNS) > 0
    else AUTO_INTERNAL_ANCHOR_COLUMNS
)

print("AUTO_INTERNAL_ANCHOR_COLUMNS:", AUTO_INTERNAL_ANCHOR_COLUMNS)
print("INTERNAL_ANCHOR_COLUMNS used:", INTERNAL_ANCHOR_COLUMNS)

if len(INTERNAL_ANCHOR_COLUMNS) == 0:
    print("[WARN] No internal anchor columns detected automatically.")

AUTO_INTERNAL_ANCHOR_COLUMNS: ['creationdate_885D', 'approvaldate_319D', 'dateactivated_425D', 'firstnonzeroinstldate_307D']
INTERNAL_ANCHOR_COLUMNS used: ['creationdate_885D', 'approvaldate_319D', 'dateactivated_425D', 'firstnonzeroinstldate_307D']


## 7. Build internal anchor wide table from raw files

Nguyên tắc:
- chỉ đọc file nào thật sự chứa anchor variable
- parse về date an toàn
- group theo `case_id`
- lấy `min()` để đại diện cho internal relationship start proxy

In [7]:

internal_anchor_hits = (
    header_with_paths
    .filter(pl.col("variable").is_in(INTERNAL_ANCHOR_COLUMNS))
    .select(["file_name", "file_path", "table_tag"])
    .unique()
    .sort("file_name")
)

print("internal_anchor_hits shape:", internal_anchor_hits.shape)
display(internal_anchor_hits)

def aggregate_internal_anchor_file(file_path: str, anchor_vars: list[str]) -> pl.DataFrame | None:
    header = read_csv_header(file_path)
    case_id_col = detect_case_id_column(header)
    if case_id_col is None:
        return None

    usable = [v for v in anchor_vars if v in header]
    if len(usable) == 0:
        return None

    read_cols = [case_id_col] + usable
    df = pl.read_csv(file_path, columns=read_cols)

    df = df.with_columns([
        pl.col(v).cast(pl.String, strict=False).alias(v)
        for v in usable
    ])

    agg_exprs = [parse_to_date_expr(v).min().alias(v) for v in usable]

    return (
        df.group_by(case_id_col)
        .agg(agg_exprs)
        .rename({case_id_col: "case_id"})
    )

internal_anchor_parts = []
for r in internal_anchor_hits.iter_rows(named=True):
    out = aggregate_internal_anchor_file(r["file_path"], INTERNAL_ANCHOR_COLUMNS)
    if out is not None:
        internal_anchor_parts.append(out)

if len(internal_anchor_parts) == 0:
    internal_anchor_wide = pl.DataFrame({"case_id": base_df["case_id"]})
else:
    internal_anchor_wide = internal_anchor_parts[0]
    for part in internal_anchor_parts[1:]:
        common_cols = [c for c in part.columns if c in internal_anchor_wide.columns and c != "case_id"]
        if common_cols:
            part = part.rename({c: f"{c}__dup" for c in common_cols})
        internal_anchor_wide = internal_anchor_wide.join(part, on="case_id", how="outer_coalesce")

    internal_anchor_wide = collapse_duplicate_suffix_columns(internal_anchor_wide, suffix="__dup")

print("internal_anchor_wide shape:", internal_anchor_wide.shape)
display(internal_anchor_wide.head(10))

internal_anchor_hits shape: (2, 3)


file_name,file_path,table_tag
str,str,str
"""train_applprev_1_0.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv""","""train_applprev_1_0"""
"""train_applprev_1_1.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_1.csv""","""train_applprev_1_1"""


internal_anchor_wide shape: (1221522, 5)


/tmp/ipykernel_17/558236329.py:52: DeprecationWarning: use of `how='outer_coalesce'` should be replaced with `how='full', coalesce=True`.
(Deprecated in version 0.20.29)
  internal_anchor_wide = internal_anchor_wide.join(part, on="case_id", how="outer_coalesce")


case_id,creationdate_885D,approvaldate_319D,dateactivated_425D,firstnonzeroinstldate_307D
i64,date,date,date,date
1652570,2018-12-13,2018-12-13,2018-12-21,2019-01-11
1406709,2008-01-05,2008-01-05,2008-01-08,2008-02-08
1285524,2016-11-24,2017-06-01,2017-06-05,2016-12-24
1519387,2019-01-05,2019-01-05,2019-02-06,2019-02-05
2531471,2014-09-04,2014-09-04,2014-09-05,2014-10-05
132201,2012-12-10,2012-12-10,2012-12-11,2013-01-11
167550,2015-11-18,2019-06-05,2019-06-07,2015-12-18
1252247,2017-09-03,2017-09-03,2017-09-21,2017-10-04
156165,2014-08-30,2014-08-30,2014-09-02,2014-09-30


## 8. Collection / delinquency candidate screening

Collection signal để tách nhóm C nên tập trung vào:
- DPD
- overdue / delinquency proxies
- willingness-to-pay stress signals

In [8]:

collection_candidates = (
    proxy_candidate_catalog
    .filter(
        pl.col("proxy_concept_final").is_in(["delinquency_risk", "willingness_to_pay", "recency_or_tenure"])
        | pl.col("variable").str.to_lowercase().str.contains("dpd|overdue|delinq|paidbefdue", literal=False)
    )
    .select([
        "variable", "description", "source_group", "proxy_concept_final",
        "stage_candidate_final", "priority_final", "leakage_risk"
    ])
    .sort(["priority_final", "variable"])
)

display(collection_candidates.head(40))

default_collection_vars = [
    "actualdpd_943P",
    "avgmaxdpdlast9m_3716943P",
    "avgdbddpdlast24m_3658932P",
    "avgdpdtolclosure24_3658938P",
]

COLLECTION_SIGNAL_VARS = [
    v for v in default_collection_vars
    if v in collection_candidates["variable"].to_list()
]

print("COLLECTION_SIGNAL_VARS:", COLLECTION_SIGNAL_VARS)

variable,description,source_group,proxy_concept_final,stage_candidate_final,priority_final,leakage_risk
str,str,str,str,str,str,str
"""actualdpd_943P""","""Days Past Due (DPD) of previous contract (actual).""","""previous_application""","""delinquency_risk""","""B_or_C""","""high""","""medium_review"""
"""actualdpdtolerance_344P""","""DPD of client with tolerance.""","""application_static""","""delinquency_risk""","""B_or_C""","""high""","""low"""
"""amtinstpaidbefduel24m_4187115A""","""Number of instalments paid before due date in the last 24 months.""","""application_static""","""willingness_to_pay""","""B_or_C""","""high""","""medium_review"""
"""avgdbddpdlast24m_3658932P""","""Average days past or before due of payment during the last 24 months.""","""application_static""","""delinquency_risk""","""C_only""","""high""","""medium_review"""
"""avgdbddpdlast3m_4187120P""","""Average days past or before due of payment during the last 3 months.""","""application_static""","""delinquency_risk""","""B_or_C""","""high""","""medium_review"""
"""collater_typofvalofguarant_298M""","""Collateral valuation type (active contract).""","""bureau""","""delinquency_risk""","""B_or_C""","""high""","""low"""
"""collater_valueofguarantee_1124L""","""Value of collateral for active contract.""","""bureau""","""delinquency_risk""","""B_or_C""","""high""","""low"""
"""collaterals_typeofguarante_669M""","""Collateral type for the active contract.""","""bureau""","""delinquency_risk""","""B_or_C""","""high""","""low"""
"""daysoverduetolerancedd_3976961L""","""Number of days that past after the due date (with tolerance).""","""application_static""","""delinquency_risk""","""B_or_C""","""high""","""medium_review"""


COLLECTION_SIGNAL_VARS: ['actualdpd_943P', 'avgdbddpdlast24m_3658932P']


## 9. Build collection signal table

In [9]:

collection_hits = (
    header_with_paths
    .filter(pl.col("variable").is_in(COLLECTION_SIGNAL_VARS))
    .select(["file_name", "file_path", "table_tag"])
    .unique()
    .sort("file_name")
)

print("collection_hits shape:", collection_hits.shape)
display(collection_hits)

def aggregate_collection_file(file_path: str, signal_vars: list[str]) -> pl.DataFrame | None:
    header = read_csv_header(file_path)
    case_id_col = detect_case_id_column(header)
    if case_id_col is None:
        return None

    usable = [v for v in signal_vars if v in header]
    if len(usable) == 0:
        return None

    df = pl.read_csv(file_path, columns=[case_id_col] + usable)

    agg_exprs = [
        safe_numeric_cast_expr(v).max().alias(v)
        for v in usable
    ]

    return (
        df.group_by(case_id_col)
        .agg(agg_exprs)
        .rename({case_id_col: "case_id"})
    )

collection_parts = []
for r in collection_hits.iter_rows(named=True):
    out = aggregate_collection_file(r["file_path"], COLLECTION_SIGNAL_VARS)
    if out is not None:
        collection_parts.append(out)

if len(collection_parts) == 0:
    collection_wide = pl.DataFrame({"case_id": base_df["case_id"]})
else:
    collection_wide = collection_parts[0]
    for part in collection_parts[1:]:
        common_cols = [c for c in part.columns if c in collection_wide.columns and c != "case_id"]
        if common_cols:
            part = part.rename({c: f"{c}__dup" for c in common_cols})
        collection_wide = collection_wide.join(part, on="case_id", how="outer_coalesce")

    collection_wide = collapse_duplicate_suffix_columns(collection_wide, suffix="__dup")

print("collection_wide shape:", collection_wide.shape)
display(collection_wide.head(10))

collection_hits shape: (4, 3)


file_name,file_path,table_tag
str,str,str
"""train_applprev_1_0.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_0.csv""","""train_applprev_1_0"""
"""train_applprev_1_1.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_applprev_1_1.csv""","""train_applprev_1_1"""
"""train_static_0_0.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_static_0_0.csv""","""train_static_0_0"""
"""train_static_0_1.csv""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_static_0_1.csv""","""train_static_0_1"""


/tmp/ipykernel_17/1270625627.py:49: DeprecationWarning: use of `how='outer_coalesce'` should be replaced with `how='full', coalesce=True`.
(Deprecated in version 0.20.29)
  collection_wide = collection_wide.join(part, on="case_id", how="outer_coalesce")


collection_wide shape: (1526659, 3)


case_id,actualdpd_943P,avgdbddpdlast24m_3658932P
i64,f64,f64
886535,0.0,null
840971,0.0,null
2580882,0.0,-6.0
24420,0.0,null
783372,0.0,-4.0
843927,0.0,null
889804,0.0,null
1245935,0.0,null
846517,0.0,null


## 10. Assign stage proxy A / B / C

Business definition:

- **A_new_no_internal_history**
  - không có internal Home Credit history proxy

- **B_behavior_no_collection**
  - có internal history
  - chưa có collection signal

- **C_collection_signal**
  - có collection / delinquency signal

In [10]:
# =========================================================
# 10. Assign stage proxy A / B / C
# =========================================================

collection_signal_exprs = [
    pl.col(v).fill_null(0)
    for v in COLLECTION_SIGNAL_VARS
    if v in collection_wide.columns
]

if len(collection_signal_exprs) == 0:
    collection_flag_expr = pl.lit(False)
else:
    collection_flag_expr = (pl.max_horizontal(collection_signal_exprs) > 0)

# FIX:
# chỉ dùng internal anchors xảy ra trước hoặc đúng decision_date
# để tránh lấy current-application dates làm prior history
valid_internal_anchor_exprs = [
    pl.when(pl.col(v) <= pl.col("decision_date")).then(pl.col(v)).otherwise(None)
    for v in INTERNAL_ANCHOR_COLUMNS
    if v in internal_anchor_wide.columns
]

if len(valid_internal_anchor_exprs) == 0:
    internal_anchor_date_expr = pl.lit(None).cast(pl.Date)
    n_internal_anchor_obs_expr = pl.lit(0)
else:
    internal_anchor_date_expr = pl.min_horizontal(valid_internal_anchor_exprs)
    n_internal_anchor_obs_expr = pl.sum_horizontal([
        pl.when(pl.col(v) <= pl.col("decision_date")).then(pl.lit(1)).otherwise(pl.lit(0))
        for v in INTERNAL_ANCHOR_COLUMNS
        if v in internal_anchor_wide.columns
    ])

stage_assignment_proxy = (
    base_df
    .join(internal_anchor_wide, on="case_id", how="left")
    .join(collection_wide, on="case_id", how="left")
    .with_columns([
        internal_anchor_date_expr.alias("internal_anchor_date"),
        n_internal_anchor_obs_expr.alias("n_internal_anchor_obs"),
        collection_flag_expr.alias("has_collection_signal"),
    ])
    .with_columns([
        (pl.col("decision_date") - pl.col("internal_anchor_date")).dt.total_days().alias("tenure_days_proxy")
    ])
    .with_columns([
        pl.when(
            (pl.col("tenure_days_proxy") < 0) | (pl.col("tenure_days_proxy") > 3650)
        )
        .then(None)
        .otherwise(pl.col("tenure_days_proxy"))
        .alias("tenure_days_proxy")
    ])
    .with_columns([
        approx_months_from_days_expr("tenure_days_proxy").alias("tenure_months_proxy")
    ])
    .with_columns([
        pl.when(pl.col("has_collection_signal") == True)
          .then(pl.lit("C_collection_signal"))
        .when(pl.col("n_internal_anchor_obs") > 0)
          .then(pl.lit("B_behavior_no_collection"))
        .otherwise(pl.lit("A_new_no_internal_history"))
        .alias("stage_final")
    ])
)

print("stage_assignment_proxy shape:", stage_assignment_proxy.shape)
display(
    stage_assignment_proxy.select([
        "case_id", "decision_date", "internal_anchor_date",
        "n_internal_anchor_obs", "has_collection_signal",
        "tenure_months_proxy", "stage_final",
        *([ "target" ] if "target" in stage_assignment_proxy.columns else [])
    ]).head(10)
)

stage_assignment_proxy shape: (1526659, 15)


case_id,decision_date,internal_anchor_date,n_internal_anchor_obs,has_collection_signal,tenure_months_proxy,stage_final,target
i64,date,date,i32,bool,f64,str,i64
0,2019-01-03,null,0,false,null,"""A_new_no_internal_history""",0
1,2019-01-03,null,0,false,null,"""A_new_no_internal_history""",0
2,2019-01-04,2013-04-03,2,false,69.06,"""B_behavior_no_collection""",0
3,2019-01-03,null,0,false,null,"""A_new_no_internal_history""",0
4,2019-01-04,null,0,false,null,"""A_new_no_internal_history""",1
5,2019-01-02,null,0,false,null,"""A_new_no_internal_history""",0
6,2019-01-03,2014-11-18,2,false,49.51,"""B_behavior_no_collection""",0
7,2019-01-03,null,0,false,null,"""A_new_no_internal_history""",0
8,2019-01-03,null,0,false,null,"""A_new_no_internal_history""",0


## 11. Stage sanity check

Check nhanh 3 thứ:
- có đủ A / B / C chưa
- tenure có còn absurd không
- target / collection rate có hợp lý không

In [11]:

stage_dist = (
    stage_assignment_proxy
    .group_by("stage_final")
    .agg([
        pl.len().alias("n_cases"),
        pl.col("case_id").n_unique().alias("n_unique_cases"),
        pl.col("n_internal_anchor_obs").mean().alias("avg_n_internal_anchor_obs"),
        pl.col("tenure_months_proxy").mean().alias("avg_tenure_m"),
        pl.col("tenure_months_proxy").median().alias("median_tenure_m"),
        pl.col("has_collection_signal").mean().alias("collection_rate"),
        *([pl.col("target").mean().alias("target_rate")] if "target" in stage_assignment_proxy.columns else [])
    ])
    .sort("n_cases", descending=True)
)

display(stage_dist)

display(
    stage_assignment_proxy
    .select(pl.col("stage_final").value_counts(sort=True))
)

display(
    stage_assignment_proxy
    .filter(pl.col("stage_final") == "A_new_no_internal_history")
    .select(["case_id", "decision_date", "internal_anchor_date", "tenure_months_proxy", "has_collection_signal"])
    .head(10)
)

display(
    stage_assignment_proxy
    .filter(pl.col("stage_final") == "B_behavior_no_collection")
    .select(["case_id", "decision_date", "internal_anchor_date", "tenure_months_proxy", "has_collection_signal"])
    .head(10)
)

display(
    stage_assignment_proxy
    .filter(pl.col("stage_final") == "C_collection_signal")
    .select(["case_id", "decision_date", "internal_anchor_date", "tenure_months_proxy", "has_collection_signal"])
    .head(10)
)

stage_final,n_cases,n_unique_cases,avg_n_internal_anchor_obs,avg_tenure_m,median_tenure_m,collection_rate,target_rate
str,u32,u32,f64,f64,f64,f64,f64
"""B_behavior_no_collection""",1085030,1085030,3.723864,47.252651,42.15,0.0,0.027805
"""A_new_no_internal_history""",318617,318617,0.0,null,null,0.0,0.022623
"""C_collection_signal""",123012,123012,3.990716,51.390461,49.58,1.0,0.086309


stage_final
struct[2]
"{""B_behavior_no_collection"",1085030}"
"{""A_new_no_internal_history"",318617}"
"{""C_collection_signal"",123012}"


case_id,decision_date,internal_anchor_date,tenure_months_proxy,has_collection_signal
i64,date,date,f64,bool
0,2019-01-03,null,null,false
1,2019-01-03,null,null,false
3,2019-01-03,null,null,false
4,2019-01-04,null,null,false
5,2019-01-02,null,null,false
7,2019-01-03,null,null,false
8,2019-01-03,null,null,false
9,2019-01-03,null,null,false
10,2019-01-03,null,null,false


case_id,decision_date,internal_anchor_date,tenure_months_proxy,has_collection_signal
i64,date,date,f64,bool
2,2019-01-04,2013-04-03,69.06,false
6,2019-01-03,2014-11-18,49.51,false
13,2019-01-03,2007-04-16,null,false
14,2019-01-03,2017-01-20,23.43,false
16,2019-01-03,2018-04-04,9.0,false
17,2019-01-03,2017-12-03,13.01,false
18,2019-01-03,2018-12-31,0.1,false
20,2019-01-03,2018-01-16,11.56,false
21,2019-01-03,2018-07-17,5.59,false


case_id,decision_date,internal_anchor_date,tenure_months_proxy,has_collection_signal
i64,date,date,f64,bool
69,2019-01-03,2018-10-01,3.09,true
292,2019-01-04,2018-10-01,3.12,true
294,2019-01-04,2017-11-24,13.34,true
367,2019-01-05,2018-12-10,0.85,true
611,2019-01-09,2018-10-10,2.99,true
835,2019-01-11,2018-10-25,2.56,true
928,2019-01-11,2018-10-29,2.43,true
936,2019-01-12,null,null,true
964,2019-01-12,2018-12-01,1.38,true


## 12. Build feature pools by stage

Ở đây vẫn giữ tinh thần từ EDA 1 nhưng làm rõ scope hơn:

- A pool: chủ yếu A_or_B
- B pool: A_or_B + B_or_C
- C pool: B_or_C + C_only + collection-heavy concepts

In [12]:

A_feature_pool = (
    proxy_candidate_catalog
    .filter(pl.col("stage_candidate_final").is_in(["A_or_B"]))
    .select("variable")
    .unique()
    .to_series()
    .to_list()
)

B_feature_pool = (
    proxy_candidate_catalog
    .filter(pl.col("stage_candidate_final").is_in(["A_or_B", "B_or_C"]))
    .select("variable")
    .unique()
    .to_series()
    .to_list()
)

C_feature_pool = (
    proxy_candidate_catalog
    .filter(
        pl.col("stage_candidate_final").is_in(["B_or_C", "C_only"])
        | pl.col("proxy_concept_final").is_in(["delinquency_risk", "willingness_to_pay"])
    )
    .select("variable")
    .unique()
    .to_series()
    .to_list()
)

print("A feature pool:", len(A_feature_pool))
print("B feature pool:", len(B_feature_pool))
print("C feature pool:", len(C_feature_pool))

A feature pool: 144
B feature pool: 208
C feature pool: 65


## 13. Lightweight raw-variable profiler

Profiler này chỉ nhằm trả lời nhanh:
- variable có đọc được không
- availability / missing rate ra sao
- numeric hay categorical
- target separation sơ bộ có gì đáng chú ý không

In [13]:

def choose_first_file_for_variable(var_name: str) -> str | None:
    hit = (
        header_with_paths
        .filter(pl.col("variable") == var_name)
        .select("file_path")
        .unique()
    )
    return hit[0, "file_path"] if hit.height > 0 else None

def profile_raw_variable(var_name: str, stage_df: pl.DataFrame) -> dict:
    file_path = choose_first_file_for_variable(var_name)
    if file_path is None:
        return {"variable": var_name, "profile_status": "missing_source"}

    header = read_csv_header(file_path)
    case_id_col = detect_case_id_column(header)
    if case_id_col is None or var_name not in header:
        return {"variable": var_name, "profile_status": "missing_case_or_column"}

    try:
        df = pl.read_csv(file_path, columns=[case_id_col, var_name])
    except Exception as e:
        return {"variable": var_name, "profile_status": f"read_error: {str(e)[:80]}"}

    agg = (
        df.group_by(case_id_col)
        .agg(pl.col(var_name).first().alias(var_name))
        .rename({case_id_col: "case_id"})
    )

    merged = stage_df.join(agg, on="case_id", how="left")
    dtype = merged.schema.get(var_name)

    out = {
        "variable": var_name,
        "file_path": file_path,
        "profile_status": "ok",
        "dtype": str(dtype),
        "n_cases": merged.height,
        "availability_rate": float(merged[var_name].is_not_null().mean()) if var_name in merged.columns else 0.0,
        "missing_rate": float(merged[var_name].is_null().mean()) if var_name in merged.columns else 1.0,
    }

    numeric_dtypes = {pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64}
    if "target" in merged.columns and var_name in merged.columns:
        if dtype in numeric_dtypes:
            tmp = merged.select([
                pl.col(var_name).cast(pl.Float64, strict=False).alias("x"),
                pl.col("target").cast(pl.Float64).alias("target")
            ])
            out["avg_value"] = float(tmp["x"].mean()) if tmp.height > 0 else None
            out["avg_value_bad"] = float(tmp.filter(pl.col("target") == 1)["x"].mean()) if tmp.height > 0 else None
            out["avg_value_good"] = float(tmp.filter(pl.col("target") == 0)["x"].mean()) if tmp.height > 0 else None
        else:
            top_vals = (
                merged
                .filter(pl.col(var_name).is_not_null())
                .group_by(var_name)
                .agg([
                    pl.len().alias("n"),
                    pl.col("target").mean().alias("bad_rate")
                ])
                .sort("n", descending=True)
                .head(5)
            )
            out["top_values_preview"] = str(top_vals.to_dicts())[:500]

    return out

## 14. Run practical EDA profiles

Để tránh notebook quá nặng, chỉ profile một tập practical:
- top A vars
- top B vars
- top C vars

In [14]:

practical_vars = list(dict.fromkeys(
    A_feature_pool[:25] + B_feature_pool[:25] + C_feature_pool[:25]
))

rows = []
for var_name in practical_vars:
    rows.append(profile_raw_variable(var_name, stage_assignment_proxy))

proxy_eda_profiles = (
    pl.DataFrame(rows)
    .with_columns([
        pl.when(pl.col("missing_rate").is_null()).then(pl.lit("review"))
          .when(pl.col("missing_rate") <= 0.60).then(pl.lit("strong"))
          .when(pl.col("missing_rate") <= 0.85).then(pl.lit("usable"))
          .otherwise(pl.lit("weak"))
          .alias("availability_bucket")
    ])
    .sort(["availability_bucket", "missing_rate", "variable"], descending=[False, False, False])
)

print("practical_vars:", len(practical_vars))
print("proxy_eda_profiles shape:", proxy_eda_profiles.shape)
display(proxy_eda_profiles.head(30))

practical_vars: 71
proxy_eda_profiles shape: (71, 12)


variable,file_path,profile_status,dtype,n_cases,availability_rate,missing_rate,top_values_preview,avg_value,avg_value_bad,avg_value_good,availability_bucket
str,str,str,str,i64,f64,f64,str,f64,f64,f64,str
"""registaddr_district_1083M""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_person_1.csv""","""ok""","""String""",1526659,1.0,0.0,"""[{'registaddr_district_1083M': 'P131_33_167', 'n': 78727, 'bad_rate': 0.030319966466396536}, {'registaddr_district_1083M': 'P197_47_166', 'n…",null,null,null,"""strong"""
"""contaddr_matchlist_1032L""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_person_1.csv""","""ok""","""Boolean""",1526659,0.993222,0.006778,"""[{'contaddr_matchlist_1032L': False, 'n': 1516311, 'bad_rate': 0.03143022770394728}]""",null,null,null,"""strong"""
"""contaddr_smempladdr_334L""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_person_1.csv""","""ok""","""Boolean""",1526659,0.993222,0.006778,"""[{'contaddr_smempladdr_334L': False, 'n': 1509416, 'bad_rate': 0.03143334905685378}, {'contaddr_smempladdr_334L': True, 'n': 6895, 'bad_rate…",null,null,null,"""strong"""
"""applicationcnt_361L""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_static_0_0.csv""","""ok""","""Float64""",1526659,0.657486,0.342514,null,0.000027,0.00013,0.000024,"""strong"""
"""clientscnt12m_3712952L""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_static_0_0.csv""","""ok""","""Float64""",1526659,0.657486,0.342514,null,0.031995,0.049946,0.031428,"""strong"""
"""clientscnt_100L""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_static_0_0.csv""","""ok""","""Float64""",1526659,0.657486,0.342514,null,0.054143,0.070055,0.05364,"""strong"""
"""clientscnt_157L""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_static_0_0.csv""","""ok""","""Float64""",1526659,0.657486,0.342514,null,0.088635,0.106921,0.088057,"""strong"""
"""clientscnt_257L""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_static_0_0.csv""","""ok""","""Float64""",1526659,0.657486,0.342514,null,0.003271,0.004458,0.003233,"""strong"""
"""clientscnt_946L""","""/kaggle/input/competitions/home-credit-credit-risk-model-stability/csv_files/train/train_static_0_0.csv""","""ok""","""Float64""",1526659,0.657486,0.342514,null,0.027727,0.025575,0.027795,"""strong"""


## 15. Build shortlist for modeling notebooks

Quy tắc đơn giản:
- `profile_status != ok` => review data
- `leakage_risk = high_review` => review time logic
- availability strong / usable => keep for FE
- còn lại => drop hoặc để sau

In [15]:

proxy_eda_shortlist = (
    proxy_candidate_catalog
    .join(
        proxy_eda_profiles.select([
            "variable", "profile_status", "availability_rate",
            "missing_rate", "availability_bucket"
        ]),
        on="variable",
        how="left"
    )
    .with_columns([
        pl.when(pl.col("profile_status").fill_null("missing") != "ok").then(pl.lit("review_data"))
          .when(pl.col("leakage_risk") == "high_review").then(pl.lit("review_time_logic"))
          .when(pl.col("availability_bucket").is_in(["strong", "usable"])).then(pl.lit("keep_for_fe"))
          .otherwise(pl.lit("drop_or_later"))
          .alias("eda_decision")
    ])
    .sort(["eda_decision", "priority_final", "variable"])
)

display(proxy_eda_shortlist.head(40))

variable,description,source_group,feature_family,proxy_concept_final,stage_candidate_final,priority_final,leakage_risk,business_validated_flag,inference_availability,review_note,n_tables,source_tables,source_files,profile_status,availability_rate,missing_rate,availability_bucket,eda_decision
str,str,str,str,str,str,str,str,bool,str,str,u32,list[str],list[str],str,f64,f64,str,str
"""amount_1115A""","""Credit amount of the active contract provided by the credit bureau.""","""bureau""","""capacity""","""credit_burden""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""active debt amount""",1,"[""train_credit_bureau_b_1""]","[""train_credit_bureau_b_1.csv""]","""ok""",0.016662,0.983338,"""weak""","""drop_or_later"""
"""amount_4917619A""","""Tax deductions amount tracked by the government registry.""","""tax_registry""","""capacity""","""capacity_to_repay""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""tax / amount proxy""",1,"[""train_tax_registry_b_1""]","[""train_tax_registry_b_1.csv""]","""ok""",0.098733,0.901267,"""weak""","""drop_or_later"""
"""amtdepositbalance_4809441A""","""Deposit balance of client.""","""other""","""capacity""","""capacity_to_repay""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""balance proxy""",1,"[""train_other_1""]","[""train_other_1.csv""]","""ok""",0.033478,0.966522,"""weak""","""drop_or_later"""
"""classificationofcontr_1114M""","""Classificiation of the active contract.""","""bureau""","""credit_burden""","""credit_burden""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""""",1,"[""train_credit_bureau_b_1""]","[""train_credit_bureau_b_1.csv""]","""ok""",0.023908,0.976092,"""weak""","""drop_or_later"""
"""collater_typofvalofguarant_298M""","""Collateral valuation type (active contract).""","""bureau""","""delinquency""","""delinquency_risk""","""B_or_C""","""high""","""low""",true,"""batch_scoring_ok""","""""",11,"[""train_credit_bureau_a_2_0"", ""train_credit_bureau_a_2_1"", … ""train_credit_bureau_a_2_9""]","[""train_credit_bureau_a_2_0.csv"", ""train_credit_bureau_a_2_1.csv"", … ""train_credit_bureau_a_2_9.csv""]","""ok""",0.102675,0.897325,"""weak""","""drop_or_later"""
"""contractmaturitydate_151D""","""End date of active contract.""","""bureau""","""credit_burden""","""credit_burden""","""A_or_B""","""high""","""medium_review""",true,"""batch_scoring_ok""","""""",1,"[""train_credit_bureau_b_1""]","[""train_credit_bureau_b_1.csv""]","""ok""",0.023813,0.976187,"""weak""","""drop_or_later"""
"""credacc_actualbalance_314A""","""Actual balance on credit account.""","""previous_application""","""credit_burden""","""credit_burden""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""""",2,"[""train_applprev_1_0"", ""train_applprev_1_1""]","[""train_applprev_1_0.csv"", ""train_applprev_1_1.csv""]","""ok""",0.021406,0.978594,"""weak""","""drop_or_later"""
"""credacc_cards_status_52L""","""Card status of the previous credit account.""","""previous_application""","""credit_burden""","""credit_burden""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""""",1,"[""train_applprev_2""]","[""train_applprev_2.csv""]","""ok""",0.04234,0.95766,"""weak""","""drop_or_later"""
"""credor_3940957M""","""Creditor's name""","""bureau""","""credit_burden""","""credit_burden""","""A_or_B""","""high""","""low""",true,"""batch_scoring_ok""","""""",1,"[""train_credit_bureau_b_1""]","[""train_credit_bureau_b_1.csv""]","""ok""",0.023908,0.976092,"""weak""","""drop_or_later"""


## 16. Create simple feature plan for notebook 3 / 4 / 5

In [16]:

feature_plan = (
    proxy_eda_shortlist
    .filter(pl.col("eda_decision") == "keep_for_fe")
    .with_columns([
        pl.when(pl.col("stage_candidate_final") == "A_or_B").then(pl.lit("A_then_B"))
          .when(pl.col("stage_candidate_final") == "B_or_C").then(pl.lit("B_then_C"))
          .when(pl.col("stage_candidate_final") == "C_only").then(pl.lit("C_only"))
          .otherwise(pl.lit("review"))
          .alias("recommended_model_scope"),

        pl.when(pl.col("proxy_concept_final") == "capacity_to_repay").then(pl.lit("max + ratio + missing_count"))
          .when(pl.col("proxy_concept_final") == "credit_burden").then(pl.lit("max + count + burden_ratio"))
          .when(pl.col("proxy_concept_final") == "willingness_to_pay").then(pl.lit("max + mean + binary_flag"))
          .when(pl.col("proxy_concept_final") == "delinquency_risk").then(pl.lit("max + mean + recent_flag"))
          .otherwise(pl.lit("simple_agg"))
          .alias("fe_idea")
    ])
    .select([
        "variable", "source_group", "proxy_concept_final", "stage_candidate_final",
        "priority_final", "recommended_model_scope", "fe_idea",
        "availability_rate", "missing_rate", "review_note"
    ])
    .sort(["recommended_model_scope", "priority_final", "variable"])
)

display(feature_plan.head(40))

variable,source_group,proxy_concept_final,stage_candidate_final,priority_final,recommended_model_scope,fe_idea,availability_rate,missing_rate,review_note
str,str,str,str,str,str,str,f64,f64,str
"""amount_4527230A""","""tax_registry""","""capacity_to_repay""","""A_or_B""","""high""","""A_then_B""","""max + ratio + missing_count""",0.299958,0.700042,"""tax / amount proxy"""
"""applicationcnt_361L""","""application_static""","""identity_instability""","""A_or_B""","""high""","""A_then_B""","""simple_agg""",0.657486,0.342514,"""repeat application signal"""
"""classificationofcontr_13M""","""bureau""","""credit_burden""","""A_or_B""","""high""","""A_then_B""","""max + count + burden_ratio""",0.219614,0.780386,""""""
"""commnoinclast6m_3546845L""","""application_static""","""capacity_to_repay""","""A_or_B""","""high""","""A_then_B""","""max + ratio + missing_count""",0.487493,0.512507,""""""
"""credacc_credlmt_575A""","""previous_application""","""credit_burden""","""A_or_B""","""high""","""A_then_B""","""max + count + burden_ratio""",0.286758,0.713242,""""""
"""currdebt_22A""","""application_static""","""capacity_to_repay""","""A_or_B""","""high""","""A_then_B""","""max + ratio + missing_count""",0.657485,0.342515,""""""
"""lastrejectcredamount_222A""","""application_static""","""capacity_to_repay""","""A_or_B""","""high""","""A_then_B""","""max + ratio + missing_count""",0.309734,0.690266,""""""
"""subjectrole_182M""","""bureau""","""credit_burden""","""A_or_B""","""high""","""A_then_B""","""max + count + burden_ratio""",0.212966,0.787034,""""""
"""clientscnt12m_3712952L""","""application_static""","""identity_instability""","""A_or_B""","""medium""","""A_then_B""","""simple_agg""",0.657486,0.342514,""""""


## 17. Export outputs

In [17]:

stage_anchor_catalog_path = WORK_DIR / "stage_anchor_catalog.parquet"
stage_assignment_path     = WORK_DIR / "stage_assignment_proxy.parquet"
proxy_profiles_path       = WORK_DIR / "proxy_eda_profiles.parquet"
proxy_shortlist_path      = WORK_DIR / "proxy_eda_shortlist.parquet"
feature_plan_path         = WORK_DIR / "feature_plan.parquet"

internal_anchor_catalog.write_parquet(stage_anchor_catalog_path)
stage_assignment_proxy.write_parquet(stage_assignment_path)
proxy_eda_profiles.write_parquet(proxy_profiles_path)
proxy_eda_shortlist.write_parquet(proxy_shortlist_path)
feature_plan.write_parquet(feature_plan_path)

print("[SAVED]", stage_anchor_catalog_path)
print("[SAVED]", stage_assignment_path)
print("[SAVED]", proxy_profiles_path)
print("[SAVED]", proxy_shortlist_path)
print("[SAVED]", feature_plan_path)

[SAVED] /kaggle/working/homecredit_proxy_notebook_02/stage_anchor_catalog.parquet
[SAVED] /kaggle/working/homecredit_proxy_notebook_02/stage_assignment_proxy.parquet
[SAVED] /kaggle/working/homecredit_proxy_notebook_02/proxy_eda_profiles.parquet
[SAVED] /kaggle/working/homecredit_proxy_notebook_02/proxy_eda_shortlist.parquet
[SAVED] /kaggle/working/homecredit_proxy_notebook_02/feature_plan.parquet


## 18. What notebook 3 should use next

Notebook tiếp theo nên dùng các artifact sau:

- `stage_assignment_proxy.parquet`
- `feature_plan.parquet`
- `proxy_eda_shortlist.parquet`

Flow đề xuất cho Notebook 3:

1. join stage vào feature mart
2. build FE riêng theo A / B / C scope
3. tránh dùng feature bị `review_time_logic`
4. export mart_A / mart_B / mart_C hoặc một mart chung có cột `stage_final`